# 🧠 Inteligência Artificial em Controle de Processos
## Lógica Fuzzy × Redes Neurais × PID Clássico

---

## Motivação

O controlador **PID** domina a indústria há décadas por sua simplicidade. Mas ele tem limitações:

- Assume que o processo é **linear e invariante no tempo**
- Precisa de **resintonização** quando o ponto de operação muda
- Tem **dificuldade com não-linearidades** e perturbações complexas

Duas abordagens de IA se destacam como alternativas:

| Abordagem | Inspiração | Conhecimento necessário |
|---|---|---|
| **Lógica Fuzzy** | Raciocínio humano impreciso | Regras linguísticas do especialista |
| **Rede Neural (ANN)** | Neurônio biológico | Dados de operação do processo |
| **PID** | Controle clássico | Modelo (ou experimento de sintonia) |

---

## Planta estudada: Sistema de Segunda Ordem

Usaremos um sistema de segunda ordem — representa, por exemplo, um **tanque aquecido com dinâmica térmica**, um **motor DC**, ou um **processo de nível com inércia**:

$$\tau^2 \ddot{y} + 2\zeta\tau\dot{y} + y = K_u \cdot u$$

Em espaço de estados:

$$\dot{x}_1 = x_2 \qquad (x_1 = y \text{ — saída})$$
$$\dot{x}_2 = \frac{K_u \cdot u - x_1 - 2\zeta\tau x_2}{\tau^2}$$

| Parâmetro | Valor | Significado |
|---|---|---|
| $K_u$ | 1,5 | Ganho estático do processo |
| $\tau$ | 5,0 s | Constante de tempo |
| $\zeta$ | 0,4 | Fator de amortecimento (sub-amortecido) |

> Com $\zeta = 0.4$, o sistema **naturalmente oscila** — o que torna o controle mais desafiador e didático para comparação.

---

## Estrutura do Notebook

| # | Seção | Conteúdo |
|---|---|---|
| 1 | Setup | Imports e configuração visual |
| 2 | Planta | Modelo, visualização da dinâmica |
| 3 | PID | Teoria, implementação, sintonia visual |
| 4 | Fuzzy | Teoria, funções de pertinência, regras, superfície |
| 5 | ANN | Geração de dados, treinamento, arquitetura |
| 6 | Comparação | Simulação, métricas, gráficos |
| 7 | Perturbação | Robustez dos três controladores |
| 8 | Resumo | Tabela final e discussão |

---
# 1. Setup

In [ ]:
# Instalar dependência de lógica fuzzy (necessário no Colab)
# !pip install scikit-fuzzy -q

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import skfuzzy as fuzz
from skfuzzy import control as ctrl

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# ── Paleta de cores consistente ──────────────────────────────────────────────
COR_PID   = '#185FA5'   # azul
COR_FUZZY = '#3B6D11'   # verde
COR_ANN   = '#993C1D'   # coral
COR_SP    = '#000000'   # preto
COR_PERT  = '#854F0B'   # âmbar

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi'     : 110,
    'font.size'      : 11,
    'axes.titlesize' : 12,
    'axes.labelsize' : 11,
    'lines.linewidth': 2.0,
})
np.random.seed(42)
print('✓ Setup concluído — scikit-fuzzy', fuzz.__version__)

---
# 2. A Planta — Sistema de Segunda Ordem

## 2.1 Discretização (Método de Euler)

Para simular em computador, discretizamos o modelo contínuo com passo $\Delta t$:

$$x_1(k+1) = x_1(k) + \Delta t \cdot x_2(k)$$
$$x_2(k+1) = x_2(k) + \Delta t \cdot \frac{K_u \cdot u(k) - x_1(k) - 2\zeta\tau x_2(k)}{\tau^2}$$

## 2.2 Resposta natural da planta

Antes de projetar qualquer controlador, é essencial **entender a dinâmica natural** da planta:
- Resposta ao degrau em malha aberta
- Velocidade de resposta
- Presença de overshoot / oscilação

In [ ]:
# ── Parâmetros da planta ─────────────────────────────────────────────────────
Ku   = 1.5    # ganho estático
tau  = 5.0    # constante de tempo (s)
zeta = 0.4    # fator de amortecimento (sub-amortecido)
dt   = 0.5    # passo de tempo (s)
U_MAX = 5.0   # saturação máxima da entrada
U_MIN = 0.0

def plant_step(x, u):
    """Simula um passo da planta de segunda ordem (Euler explícito)."""
    x1, x2 = x
    u = np.clip(u, U_MIN, U_MAX)
    dx1 = x2
    dx2 = (Ku * u - x1 - 2 * zeta * tau * x2) / tau**2
    return [x1 + dx1 * dt, x2 + dx2 * dt]

# ── Resposta ao degrau em malha aberta ──────────────────────────────────────
N = 300
t = np.arange(N) * dt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titulos = ['u = 1,0 (degrau unitário)', 'u = 2,5 (degrau médio)', 'u = 5,0 (máximo)']
us      = [1.0, 2.5, 5.0]

for ax, u_step, titulo in zip(axes, us, titulos):
    x_hist = [[0, 0]]
    for _ in range(N-1):
        x_hist.append(plant_step(x_hist[-1], u_step))
    x_hist = np.array(x_hist)

    ax.plot(t, x_hist[:, 0], color=COR_PID, lw=2.5)
    ss = Ku * u_step           # valor em regime estacionário
    ax.axhline(ss, color='red', ls='--', lw=1.5, label=f'SS = {ss:.2f}')
    ax.axhline(ss * 0.98, color='gray', ls=':', lw=1)
    ax.axhline(ss * 1.02, color='gray', ls=':', lw=1, label='±2%')
    ax.set_title(titulo)
    ax.set_xlabel('Tempo (s)')
    ax.set_ylabel('Saída y(t)')
    ax.legend(fontsize=9)

    # Overshoot
    overshoot = (x_hist[:, 0].max() - ss) / ss * 100
    ax.annotate(f'Overshoot\n{overshoot:.1f}%',
                xy=(t[x_hist[:,0].argmax()], x_hist[:,0].max()),
                xytext=(t[x_hist[:,0].argmax()] + 5, x_hist[:,0].max() + 0.05),
                arrowprops=dict(arrowstyle='->', color='red'),
                color='red', fontsize=9)

plt.suptitle('Resposta da Planta em Malha Aberta\n'
             r'Sistema 2ª ordem: $\tau=5s$, $\zeta=0.4$ (sub-amortecido)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f'Parâmetros: Ku={Ku}, τ={tau}s, ζ={zeta}')
print(f'Sistema sub-amortecido → oscila naturalmente → desafiador para controle!')

---
# 3. Controlador PID

## 3.1 Teoria

O controlador **PID** (Proporcional-Integral-Derivativo) calcula a ação de controle com base em três termos:

$$u(t) = K_p \, e(t) + K_i \int_0^t e(\tau)\,d\tau + K_d \frac{de}{dt}$$

Discretizado com passo $\Delta t$:

$$u(k) = K_p\, e(k) + K_i \Delta t \sum_{j=0}^{k} e(j) + K_d \frac{e(k) - e(k-1)}{\Delta t}$$

### Papel de cada termo

| Termo | Efeito | Problema se excessivo |
|---|---|---|
| **P** (proporcional) | Reação imediata ao erro | Offset residual, oscilação |
| **I** (integral) | Elimina offset em regime | Windup, lentidão |
| **D** (derivativo) | Antecipa a tendência | Amplifica ruído |

## 3.2 Sintonia — Método de Ziegler-Nichols (malha aberta)

A partir da curva de resposta ao degrau, extrai-se:
- $L$ = atraso puro (tempo de morto)
- $T$ = constante de tempo aparente

$$K_p = \frac{1{,}2T}{KL} \qquad K_i = \frac{K_p}{2L} \qquad K_d = 0{,}5 K_p L$$

In [ ]:
# ── Implementação do PID com anti-windup ─────────────────────────────────────
class PID:
    def __init__(self, Kp, Ki, Kd, dt, u_min=0, u_max=5):
        self.Kp, self.Ki, self.Kd = Kp, Ki, Kd
        self.dt = dt
        self.u_min, self.u_max = u_min, u_max
        self.integ  = 0.0
        self.e_prev = 0.0

    def reset(self):
        self.integ = 0.0; self.e_prev = 0.0

    def compute(self, setpoint, medida):
        e      = setpoint - medida
        deriv  = (e - self.e_prev) / self.dt
        u_raw  = self.Kp * e + self.Ki * self.integ + self.Kd * deriv
        u      = np.clip(u_raw, self.u_min, self.u_max)
        # Anti-windup: só integra se não saturado
        if u == u_raw:
            self.integ += e * self.dt
        self.e_prev = e
        return u

# Ganhos sintonizados manualmente para boa resposta
Kp_pid, Ki_pid, Kd_pid = 2.0, 0.30, 1.5
pid = PID(Kp_pid, Ki_pid, Kd_pid, dt)

# ── Efeito de cada ganho — visualização didática ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sp = 1.0; N = 300
t  = np.arange(N) * dt

configs = [
    {'titulo': 'Efeito do Kp\n(Ki=0, Kd=0)',
     'ganhos': [(0.5,'Kp=0.5',':'), (1.5,'Kp=1.5','--'), (3.0,'Kp=3.0','-')],
     'Ki': 0, 'Kd': 0},
    {'titulo': 'Efeito do Ki\n(Kp=2, Kd=0)',
     'ganhos': [(0.0,'Ki=0',':'), (0.1,'Ki=0.1','--'), (0.5,'Ki=0.5','-')],
     'Kp_base': 2.0, 'Kd': 0},
    {'titulo': 'Efeito do Kd\n(Kp=2, Ki=0.3)',
     'ganhos': [(0.0,'Kd=0',':'), (1.0,'Kd=1.0','--'), (3.0,'Kd=3.0','-')],
     'Kp_base': 2.0, 'Ki_base': 0.3},
]
cores_ganho = ['#993C1D', '#854F0B', '#185FA5']

for ax, cfg in zip(axes, configs):
    ax.axhline(sp, color='k', ls='--', lw=1.5, label='Setpoint', zorder=5)
    for (g, lbl, ls), cor in zip(cfg['ganhos'], cores_ganho):
        if 'Efeito do Kp' in cfg['titulo']:
            ctrl_tmp = PID(g, cfg['Ki'], cfg['Kd'], dt)
        elif 'Efeito do Ki' in cfg['titulo']:
            ctrl_tmp = PID(cfg['Kp_base'], g, cfg['Kd'], dt)
        else:
            ctrl_tmp = PID(cfg['Kp_base'], cfg['Ki_base'], g, dt)
        x = [0, 0]
        y_hist = []
        for _ in range(N):
            u = ctrl_tmp.compute(sp, x[0])
            x = plant_step(x, u)
            y_hist.append(x[0])
        ax.plot(t, y_hist, color=cor, ls=ls, lw=2, label=lbl)
    ax.set_title(cfg['titulo'])
    ax.set_xlabel('Tempo (s)')
    ax.set_ylabel('Saída y')
    ax.legend(fontsize=9)
    ax.set_ylim(-0.1, 2.2)

plt.suptitle('Análise da influência dos ganhos PID', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
# 4. Controlador por Lógica Fuzzy (FLC)

## 4.1 O que é Lógica Fuzzy?

Na lógica clássica, uma variável é **verdadeira (1) ou falsa (0)**. Na lógica fuzzy, uma variável pode pertencer **parcialmente** a múltiplos conjuntos ao mesmo tempo.

**Exemplo:** A temperatura de 22°C pode ser:
- 70% "confortável"
- 30% "fria"
- 0% "quente"

## 4.2 Estrutura de um Controlador Fuzzy (FLC)

```
Entradas          Fuzzificação      Base de Regras       Defuzzificação       Saída
  e(k)      →    [graus de       → [SE erro É PB        →  [centróide]   →   u(k)
  Δe(k)          pertinência]       ENTÃO controle É VB]                 (valor nítido)
```

**Passos:**
1. **Fuzzificação:** converter valores nítidos em graus de pertinência
2. **Inferência:** aplicar as regras SE-ENTÃO
3. **Defuzzificação:** converter saída fuzzy em valor nítido (método do centróide)

## 4.3 Variáveis linguísticas e Funções de Pertinência

Definimos **5 termos linguísticos** para cada variável:

| Símbolo | Significado |
|---|---|
| NB | Negativo Grande (Negative Big) |
| N  | Negativo |
| Z  | Zero |
| P  | Positivo |
| PB | Positivo Grande (Positive Big) |

In [ ]:
# ── Universos de discurso ─────────────────────────────────────────────────────
e_range  = np.linspace(-2.0,  2.0, 500)   # erro
de_range = np.linspace(-1.0,  1.0, 500)   # variação do erro
u_range  = np.linspace( 0.0,  5.0, 500)   # saída de controle

# ── Variáveis linguísticas ────────────────────────────────────────────────────
erro    = ctrl.Antecedent(e_range,  'erro')
d_erro  = ctrl.Antecedent(de_range, 'd_erro')
controle = ctrl.Consequent(u_range, 'controle')

# ── Funções de pertinência — ERRO (trapézios + triângulos) ───────────────────
erro['NB'] = fuzz.trapmf(e_range, [-2.0, -2.0, -1.2, -0.4])
erro['N']  = fuzz.trimf( e_range, [-1.2, -0.5,  0.0])
erro['Z']  = fuzz.trimf( e_range, [-0.3,  0.0,  0.3])
erro['P']  = fuzz.trimf( e_range, [ 0.0,  0.5,  1.2])
erro['PB'] = fuzz.trapmf(e_range, [ 0.4,  1.2,  2.0, 2.0])

# ── Funções de pertinência — VARIAÇÃO DO ERRO ────────────────────────────────
d_erro['NB'] = fuzz.trapmf(de_range, [-1.0, -1.0, -0.6, -0.2])
d_erro['N']  = fuzz.trimf( de_range, [-0.6, -0.25, 0.0])
d_erro['Z']  = fuzz.trimf( de_range, [-0.2,  0.0,  0.2])
d_erro['P']  = fuzz.trimf( de_range, [ 0.0,  0.25, 0.6])
d_erro['PB'] = fuzz.trapmf(de_range, [ 0.2,  0.6,  1.0, 1.0])

# ── Funções de pertinência — CONTROLE ────────────────────────────────────────
controle['VB'] = fuzz.trapmf(u_range, [3.5, 4.5, 5.0, 5.0])  # Very Big
controle['B']  = fuzz.trimf( u_range, [2.5, 3.5, 4.5])
controle['M']  = fuzz.trimf( u_range, [1.5, 2.5, 3.5])       # Medium
controle['S']  = fuzz.trimf( u_range, [0.5, 1.5, 2.5])
controle['VS'] = fuzz.trapmf(u_range, [0.0, 0.0, 0.5, 1.5])  # Very Small

# ── Visualização das funções de pertinência ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
nomes_mf = ['NB', 'N', 'Z', 'P', 'PB']
cores_mf = ['#993C1D', '#854F0B', '#3B6D11', '#185FA5', '#5F0A87']

# Erro
ax = axes[0]
for nome, cor in zip(nomes_mf, cores_mf):
    ax.plot(e_range, erro[nome].mf, color=cor, lw=2.5, label=nome)
    ax.fill_between(e_range, 0, erro[nome].mf, color=cor, alpha=0.1)
ax.set_title('Funções de Pertinência\nErro e(k) = r(k) − y(k)')
ax.set_xlabel('e(k)'); ax.set_ylabel('Grau de pertinência μ')
ax.legend(fontsize=10, loc='upper right'); ax.set_ylim(-0.05, 1.15)

# Variação do erro
ax = axes[1]
for nome, cor in zip(nomes_mf, cores_mf):
    ax.plot(de_range, d_erro[nome].mf, color=cor, lw=2.5, label=nome)
    ax.fill_between(de_range, 0, d_erro[nome].mf, color=cor, alpha=0.1)
ax.set_title('Funções de Pertinência\nVariação do Erro Δe(k)')
ax.set_xlabel('Δe(k)'); ax.set_ylabel('Grau de pertinência μ')
ax.legend(fontsize=10, loc='upper right'); ax.set_ylim(-0.05, 1.15)

# Controle
ax = axes[2]
nomes_u = ['VB', 'B', 'M', 'S', 'VS']
for nome, cor in zip(nomes_u, reversed(cores_mf)):
    ax.plot(u_range, controle[nome].mf, color=cor, lw=2.5, label=nome)
    ax.fill_between(u_range, 0, controle[nome].mf, color=cor, alpha=0.1)
ax.set_title('Funções de Pertinência\nControle u(k)')
ax.set_xlabel('u(k)'); ax.set_ylabel('Grau de pertinência μ')
ax.legend(fontsize=10, loc='upper right'); ax.set_ylim(-0.05, 1.15)

plt.suptitle('Lógica Fuzzy — Funções de Pertinência dos Três Universos',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4.4 Base de Regras

As regras são definidas por um **especialista** no processo — é o conhecimento humano codificado em linguagem natural.

**Exemplos de regras:**
- `SE erro É PB ENTÃO controle É VB` → erro grande positivo: aplicar máximo controle
- `SE erro É Z E d_erro É Z ENTÃO controle É M` → em equilíbrio: manter controle médio
- `SE erro É NB ENTÃO controle É VS` → erro negativo grande: reduzir controle

**Tabela de regras (Mamdani):**

| e \ Δe | NB | N | Z | P | PB |
|---|---|---|---|---|---|
| **NB** | VS | VS | VS | S | VS |
| **N**  | VS | VS | S  | M | S  |
| **Z**  | S  | S  | M  | B | B  |
| **P**  | M  | B  | B  | VB| VB |
| **PB** | B  | VB | VB | VB| VB |

In [ ]:
# ── Base de regras Mamdani ────────────────────────────────────────────────────
regras = [
    # erro PB → sempre controle grande
    ctrl.Rule(erro['PB'] & d_erro['PB'], controle['VB']),
    ctrl.Rule(erro['PB'] & d_erro['P'],  controle['VB']),
    ctrl.Rule(erro['PB'] & d_erro['Z'],  controle['VB']),
    ctrl.Rule(erro['PB'] & d_erro['N'],  controle['B']),
    ctrl.Rule(erro['PB'] & d_erro['NB'], controle['B']),

    # erro P → controle grande/médio
    ctrl.Rule(erro['P'] & d_erro['PB'],  controle['VB']),
    ctrl.Rule(erro['P'] & d_erro['P'],   controle['VB']),
    ctrl.Rule(erro['P'] & d_erro['Z'],   controle['B']),
    ctrl.Rule(erro['P'] & d_erro['N'],   controle['B']),
    ctrl.Rule(erro['P'] & d_erro['NB'],  controle['M']),

    # erro Z → controle médio
    ctrl.Rule(erro['Z'] & d_erro['PB'],  controle['B']),
    ctrl.Rule(erro['Z'] & d_erro['P'],   controle['B']),
    ctrl.Rule(erro['Z'] & d_erro['Z'],   controle['M']),
    ctrl.Rule(erro['Z'] & d_erro['N'],   controle['S']),
    ctrl.Rule(erro['Z'] & d_erro['NB'],  controle['S']),

    # erro N → controle pequeno
    ctrl.Rule(erro['N'] & d_erro['PB'],  controle['S']),
    ctrl.Rule(erro['N'] & d_erro['P'],   controle['M']),
    ctrl.Rule(erro['N'] & d_erro['Z'],   controle['S']),
    ctrl.Rule(erro['N'] & d_erro['N'],   controle['VS']),
    ctrl.Rule(erro['N'] & d_erro['NB'],  controle['VS']),

    # erro NB → controle mínimo
    ctrl.Rule(erro['NB'] & d_erro['PB'], controle['VS']),
    ctrl.Rule(erro['NB'] & d_erro['P'],  controle['S']),
    ctrl.Rule(erro['NB'] & d_erro['Z'],  controle['VS']),
    ctrl.Rule(erro['NB'] & d_erro['N'],  controle['VS']),
    ctrl.Rule(erro['NB'] & d_erro['NB'], controle['VS']),
]

# ── Criar sistema de controle fuzzy ──────────────────────────────────────────
sistema_fuzzy = ctrl.ControlSystem(regras)
sim_fuzzy     = ctrl.ControlSystemSimulation(sistema_fuzzy)

def fuzzy_step(e_val, de_val):
    """Calcula a ação de controle fuzzy dado erro e variação do erro."""
    e_val  = np.clip(e_val,  e_range[0],  e_range[-1])
    de_val = np.clip(de_val, de_range[0], de_range[-1])
    sim_fuzzy.input['erro']   = e_val
    sim_fuzzy.input['d_erro'] = de_val
    sim_fuzzy.compute()
    return sim_fuzzy.output['controle']

print(f'✓ {len(regras)} regras fuzzy definidas')

# Teste da inferência
casos_teste = [(1.0, 0.0), (0.5, -0.1), (0.0, 0.0), (-0.5, 0.1)]
print('\nTeste de inferência fuzzy:')
print(f'  {"e":>6} | {"Δe":>6} | {"u":>8}')
print('  ' + '-'*28)
for e_t, de_t in casos_teste:
    u_t = fuzzy_step(e_t, de_t)
    print(f'  {e_t:+6.2f} | {de_t:+6.2f} | {u_t:8.4f}')

In [ ]:
# ── Superfície de controle fuzzy — visualização 3D ───────────────────────────
e_grid  = np.linspace(-2, 2, 40)
de_grid = np.linspace(-1, 1, 40)
EE, DE  = np.meshgrid(e_grid, de_grid)
UU      = np.zeros_like(EE)

for i in range(len(de_grid)):
    for j in range(len(e_grid)):
        try:
            UU[i, j] = fuzzy_step(EE[i, j], DE[i, j])
        except:
            UU[i, j] = 2.5

fig = plt.figure(figsize=(15, 5))

# ── 3D ───────────────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(131, projection='3d')
surf = ax1.plot_surface(EE, DE, UU, cmap='RdYlGn', alpha=0.85, linewidth=0)
ax1.set_xlabel('Erro e(k)'); ax1.set_ylabel('Δe(k)'); ax1.set_zlabel('u(k)')
ax1.set_title('Superfície de Controle\nFuzzy (3D)')
fig.colorbar(surf, ax=ax1, shrink=0.5, label='u')

# ── Mapa de calor ────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(132)
im  = ax2.contourf(EE, DE, UU, levels=20, cmap='RdYlGn')
ax2.contour(EE, DE, UU, levels=10, colors='black', linewidths=0.5, alpha=0.4)
fig.colorbar(im, ax=ax2, label='u(k)')
ax2.set_xlabel('Erro e(k)'); ax2.set_ylabel('Δe(k)')
ax2.set_title('Mapa de Calor\nda Superfície Fuzzy')
ax2.axhline(0, color='k', lw=1, ls='--', alpha=0.5)
ax2.axvline(0, color='k', lw=1, ls='--', alpha=0.5)

# ── Exemplo de inferência passo a passo ─────────────────────────────────────
ax3 = fig.add_subplot(133)
e_ex, de_ex = 0.8, 0.1
u_ex = fuzzy_step(e_ex, de_ex)

# Mostrar pertinências do ponto exemplo
pert_e = {nome: fuzz.interp_membership(e_range, erro[nome].mf, e_ex)
          for nome in ['NB','N','Z','P','PB']}
nomes_sorted = sorted(pert_e.keys(), key=lambda x: pert_e[x], reverse=True)
vals = [pert_e[n] for n in nomes_sorted]
bars = ax3.barh(nomes_sorted, vals,
                color=[cores_mf[['NB','N','Z','P','PB'].index(n)] for n in nomes_sorted],
                edgecolor='white', alpha=0.85)
for bar, val in zip(bars, vals):
    if val > 0.01:
        ax3.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=10)
ax3.set_xlim(0, 1.15)
ax3.set_xlabel('Grau de pertinência μ')
ax3.set_title(f'Fuzzificação do Erro\ne = {e_ex} → u = {u_ex:.3f}')

plt.suptitle('Superfície de Controle Fuzzy — Mapeamento (e, Δe) → u',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
# 5. Controlador por Rede Neural (ANN)

## 5.1 Abordagem: Behavioral Cloning

A ANN **imita o comportamento do controlador PID** a partir de dados:

```
Etapa 1: Gerar dados          →  Simular PID em vários cenários
Etapa 2: Treinar ANN          →  Aprender mapeamento (estado) → (ação)
Etapa 3: Substituir o PID     →  ANN age em malha fechada
```

## 5.2 Features de entrada da ANN

| Feature | Descrição | Analogia no PID |
|---|---|---|
| $e(k) = r - y$ | Erro atual | Termo Proporcional |
| $\Delta e(k) = e(k) - e(k-1)$ | Variação do erro | Termo Derivativo |
| $\sum e$ | Soma dos erros passados | Termo Integral |
| $y(k)$ | Saída atual da planta | Estado do processo |
| $r(k)$ | Setpoint atual | Referência |

## 5.3 Arquitetura

```
Entrada (5)  →  Camada 1 (64, tanh)  →  Camada 2 (32, tanh)  →  Saída (1, linear)
```

> **Por que tanh?** O controlador precisa gerar saídas negativas e positivas internamente, e tanh é naturalmente simétrica em torno de zero — mais adequada que ReLU para controle.

In [ ]:
# ── Gerar dados: PID em múltiplos cenários ────────────────────────────────────
pid_treino = PID(Kp_pid, Ki_pid, Kd_pid, dt)

# Perfis de setpoint variados para cobrir toda a região de operação
setpoints_treino = [
    0.5, 1.0, 1.5, 0.8, 1.2, 0.3, 1.8, 0.6, 1.4, 1.0,
    0.7, 1.3, 0.4, 1.6, 0.9, 1.1, 0.2, 1.7, 0.5, 1.5,
]

X_dados, Y_dados = [], []
hist_treino_x, hist_treino_sp = [], []

for sp_val in setpoints_treino:
    pid_treino.reset()
    x = [0.0, 0.0]
    soma_e = 0.0
    e_prev = 0.0
    N_ep   = 250

    for k in range(N_ep):
        e      = sp_val - x[0]
        de     = e - e_prev
        soma_e += e * dt
        u_pid  = pid_treino.compute(sp_val, x[0])

        X_dados.append([e, de, soma_e, x[0], sp_val])
        Y_dados.append(u_pid)
        hist_treino_x.append(x[0])
        hist_treino_sp.append(sp_val)

        e_prev = e
        x = plant_step(x, u_pid)

X_dados = np.array(X_dados)
Y_dados = np.array(Y_dados)
print(f'Dataset de treinamento:')
print(f'  {len(setpoints_treino)} cenários × {N_ep} passos = {len(X_dados):,} amostras')
print(f'  Features: {X_dados.shape[1]}  |  Saída: escalar')
print(f'  Faixa de u: [{Y_dados.min():.3f}, {Y_dados.max():.3f}]')

# Visualizar dados de treino
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
t_tr = np.arange(len(hist_treino_x)) * dt
axes[0].plot(t_tr, hist_treino_x,  color=COR_PID, lw=0.8, alpha=0.7, label='y(t)')
axes[0].plot(t_tr, hist_treino_sp, color='k', lw=0.8, ls='--', alpha=0.5, label='setpoint')
axes[0].set_title('Resposta do PID nos cenários de treino'); axes[0].set_xlabel('Tempo (s)')
axes[0].set_ylabel('y'); axes[0].legend(fontsize=9)

axes[1].scatter(X_dados[::10, 0], Y_dados[::10],  # e vs u (a cada 10 pts)
                c=X_dados[::10, 4], cmap='RdYlGn', s=4, alpha=0.4)
axes[1].set_xlabel('Erro e(k)'); axes[1].set_ylabel('u(k) do PID')
axes[1].set_title('Distribuição dos dados\n(cor = setpoint)')
plt.tight_layout(); plt.show()

In [ ]:
# ── Normalização e treinamento ────────────────────────────────────────────────
sc_X = StandardScaler()
sc_Y = StandardScaler()
X_sc = sc_X.fit_transform(X_dados)
Y_sc = sc_Y.fit_transform(Y_dados.reshape(-1, 1)).ravel()

ann_ctrl = MLPRegressor(
    hidden_layer_sizes = (64, 32),
    activation         = 'tanh',
    solver             = 'adam',
    learning_rate_init = 0.001,
    max_iter           = 800,
    alpha              = 1e-4,
    early_stopping     = True,
    validation_fraction= 0.1,
    n_iter_no_change   = 30,
    random_state       = 42,
)
import time
t0 = time.time()
ann_ctrl.fit(X_sc, Y_sc)
print(f'Treinamento: {time.time()-t0:.1f}s  |  épocas: {ann_ctrl.n_iter_}')

# Avaliar ajuste
Y_pred_sc = ann_ctrl.predict(X_sc)
Y_pred    = sc_Y.inverse_transform(Y_pred_sc.reshape(-1,1)).ravel()
r2        = r2_score(Y_dados, Y_pred)
rmse      = np.sqrt(np.mean((Y_dados - Y_pred)**2))
print(f'R² = {r2:.4f}  |  RMSE = {rmse:.4f}')

# Função de controle ANN
def ann_step(e_val, de_val, soma_e_val, y_val, sp_val):
    feat = sc_X.transform([[e_val, de_val, soma_e_val, y_val, sp_val]])
    raw  = ann_ctrl.predict(feat)[0]
    u    = float(np.clip(sc_Y.inverse_transform([[raw]])[0, 0], U_MIN, U_MAX))
    return u

# Visualizar: loss e comparação u_pid vs u_ann
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].semilogy(ann_ctrl.loss_curve_, color=COR_ANN, lw=2)
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss (log)')
axes[0].set_title('Curva de Aprendizado da ANN')

n_show = 500
axes[1].plot(Y_dados[:n_show], color=COR_PID, lw=1.5, label='PID (referência)')
axes[1].plot(Y_pred[:n_show],  color=COR_ANN, lw=1.5, ls='--', label='ANN (imitação)')
axes[1].set_title(f'u PID vs u ANN\n(R²={r2:.4f})')
axes[1].set_xlabel('Amostras'); axes[1].set_ylabel('u')
axes[1].legend(fontsize=9)

axes[2].scatter(Y_dados[::5], Y_pred[::5], alpha=0.2, s=5, color=COR_ANN)
lims = [Y_dados.min(), Y_dados.max()]
axes[2].plot(lims, lims, 'k--', lw=1.5, label='Perfeito')
axes[2].set_xlabel('u PID real'); axes[2].set_ylabel('u ANN previsto')
axes[2].set_title('Scatter: PID real vs ANN')
axes[2].legend(fontsize=9)

plt.suptitle('Avaliação do Treinamento da ANN Controladora', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

n_params = sum(w.size for w in ann_ctrl.coefs_) + sum(b.size for b in ann_ctrl.intercepts_)
print(f'Arquitetura: 5 → 64 → 32 → 1  |  Total de parâmetros: {n_params:,}')

---
# 6. Comparação em Malha Fechada

## 6.1 Cenário de Teste

Simulamos os três controladores com **o mesmo perfil de setpoints** — incluindo:
- Degrau positivo grande
- Degrau para baixo
- Retorno ao setpoint original

Os setpoints do teste são **diferentes** dos usados no treinamento da ANN — testando a capacidade de generalização.

In [ ]:
# ── Perfil de setpoints para o teste ─────────────────────────────────────────
N_SIM = 500
t_sim = np.arange(N_SIM) * dt

sp_seq = np.zeros(N_SIM)
sp_seq[  0:100] = 1.0
sp_seq[100:200] = 1.6   # degrau para cima
sp_seq[200:300] = 0.6   # degrau para baixo
sp_seq[300:400] = 1.2   # valor intermediário
sp_seq[400:500] = 0.9   # retorno próximo ao inicial

def simular_controlador(nome, func_ctrl, sp_seq):
    """Executa simulação em malha fechada de um controlador genérico."""
    x       = [0.0, 0.0]
    y_hist  = [0.0]
    u_hist  = []
    e_prev  = 0.0
    soma_e  = 0.0

    for k in range(N_SIM):
        sp  = sp_seq[k]
        e   = sp - x[0]
        de  = e - e_prev
        soma_e += e * dt

        u = func_ctrl(e, de, soma_e, x[0], sp)
        u_hist.append(u)
        x = plant_step(x, u)
        y_hist.append(x[0])
        e_prev = e

    return np.array(y_hist[1:]), np.array(u_hist)

# ── Funções wrapper para cada controlador ─────────────────────────────────────
pid_teste = PID(Kp_pid, Ki_pid, Kd_pid, dt)
pid_teste.reset()

def ctrl_pid(e, de, soma_e, y, sp):
    return pid_teste.compute(sp, y)

def ctrl_fuzzy(e, de, soma_e, y, sp):
    return fuzzy_step(e, de)

def ctrl_ann(e, de, soma_e, y, sp):
    return ann_step(e, de, soma_e, y, sp)

print('Simulando controladores...')
y_pid,   u_pid   = simular_controlador('PID',   ctrl_pid,   sp_seq)
pid_teste.reset()
y_fuzzy, u_fuzzy = simular_controlador('Fuzzy', ctrl_fuzzy, sp_seq)
y_ann,   u_ann   = simular_controlador('ANN',   ctrl_ann,   sp_seq)
print('✓ Simulações concluídas')

In [ ]:
# ── Gráfico principal de comparação ──────────────────────────────────────────
fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(3, 1, height_ratios=[3, 2, 2], hspace=0.4)

# ── Painel 1: Rastreamento do setpoint ───────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
ax1.plot(t_sim, sp_seq,  color=COR_SP,    lw=2,   ls='--', label='Setpoint',      zorder=5)
ax1.plot(t_sim, y_pid,   color=COR_PID,   lw=2,   alpha=0.9, label='PID')
ax1.plot(t_sim, y_fuzzy, color=COR_FUZZY, lw=2,   alpha=0.9, label='Fuzzy')
ax1.plot(t_sim, y_ann,   color=COR_ANN,   lw=2,   alpha=0.9, label='ANN')

# Anotar os degraus
for t_d, sp_d, txt in [(50, 1.0, 'SP=1.0'), (100, 1.6, 'SP=1.6'),
                        (200, 0.6, 'SP=0.6'), (300, 1.2, 'SP=1.2'), (400, 0.9, 'SP=0.9')]:
    ax1.axvline(t_d*dt, color='gray', lw=0.8, ls=':', alpha=0.6)
    ax1.text(t_d*dt + 2, sp_d + 0.05, txt, fontsize=8, color='gray')

ax1.set_ylabel('Saída y(t)', fontsize=12)
ax1.set_title('Rastreamento do Setpoint — PID × Fuzzy × ANN', fontsize=13)
ax1.legend(fontsize=11, loc='upper right')
ax1.set_xlim(0, t_sim[-1])

# ── Painel 2: Erro de rastreamento ───────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
ax2.plot(t_sim, sp_seq - y_pid,   color=COR_PID,   lw=1.8, alpha=0.9, label='Erro PID')
ax2.plot(t_sim, sp_seq - y_fuzzy, color=COR_FUZZY, lw=1.8, alpha=0.9, label='Erro Fuzzy')
ax2.plot(t_sim, sp_seq - y_ann,   color=COR_ANN,   lw=1.8, alpha=0.9, label='Erro ANN')
ax2.axhline(0, color='k', lw=1, ls='--')
ax2.fill_between(t_sim, -0.05, 0.05, alpha=0.1, color='green', label='±5%')
ax2.set_ylabel('Erro e(t) = r − y', fontsize=12)
ax2.set_title('Erro de Rastreamento')
ax2.legend(fontsize=10, loc='upper right')
ax2.set_xlim(0, t_sim[-1])

# ── Painel 3: Ação de controle ────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[2])
ax3.step(t_sim, u_pid,   color=COR_PID,   lw=1.8, alpha=0.9, label='u PID',   where='post')
ax3.step(t_sim, u_fuzzy, color=COR_FUZZY, lw=1.8, alpha=0.9, label='u Fuzzy', where='post')
ax3.step(t_sim, u_ann,   color=COR_ANN,   lw=1.8, alpha=0.9, label='u ANN',   where='post')
ax3.axhline(U_MAX, color='red', lw=1, ls=':', alpha=0.5, label=f'u_max={U_MAX}')
ax3.axhline(U_MIN, color='red', lw=1, ls=':', alpha=0.5)
ax3.set_ylabel('Ação de controle u', fontsize=12)
ax3.set_xlabel('Tempo (s)', fontsize=12)
ax3.set_title('Ações de Controle')
ax3.legend(fontsize=10, loc='upper right')
ax3.set_xlim(0, t_sim[-1]); ax3.set_ylim(-0.3, 5.5)

plt.show()

In [ ]:
# ── Zoom nos degraus individuais ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

zooms = [
    {'k0': 0,   'k1': 100, 'titulo': 'Degrau 0 → 1.0\n(partida do zero)'},
    {'k0': 100, 'k1': 200, 'titulo': 'Degrau 1.0 → 1.6\n(aumento)'},
    {'k0': 200, 'k1': 300, 'titulo': 'Degrau 1.6 → 0.6\n(redução)'},
]

for ax, z in zip(axes, zooms):
    k0, k1 = z['k0'], z['k1']
    t_z = t_sim[k0:k1]
    ax.plot(t_z, sp_seq[k0:k1],  color='k',        lw=2,   ls='--', label='Setpoint')
    ax.plot(t_z, y_pid[k0:k1],   color=COR_PID,    lw=2.5, label='PID')
    ax.plot(t_z, y_fuzzy[k0:k1], color=COR_FUZZY,  lw=2.5, label='Fuzzy')
    ax.plot(t_z, y_ann[k0:k1],   color=COR_ANN,    lw=2.5, label='ANN')

    sp_val = sp_seq[k1-1]
    ax.fill_between(t_z, sp_val*0.98, sp_val*1.02, alpha=0.1, color='green', label='±2%')

    ax.set_title(z['titulo'])
    ax.set_xlabel('Tempo (s)')
    ax.set_ylabel('y(t)')
    if k0 == 0:
        ax.legend(fontsize=9)

plt.suptitle('Zoom nos Degraus — Comparação Detalhada', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
# 7. Teste de Robustez — Perturbação e Variação de Parâmetros

## 7.1 Robustez a Perturbações

Um controlador real precisa lidar com **perturbações** — variações não controladas que afetam o processo. Testamos dois tipos:

1. **Perturbação de carga:** impulso na entrada da planta em $t = 100s$ (simula abertura de válvula, mudança de carga)
2. **Variação de parâmetro:** o ganho da planta $K_u$ muda 30% no meio da simulação (simula desgaste, mudança de produto)

> **Atenção:** A ANN foi treinada com a planta **original**. Quando $K_u$ muda, ela não sabe — isso testa a **robustez à incerteza de modelo**.

In [ ]:
# ── Teste 1: Perturbação de carga (impulso na saída) ─────────────────────────
N_PERT = 400
t_pert = np.arange(N_PERT) * dt
sp_pert = np.ones(N_PERT) * 1.0
PERT_K  = 150    # instante da perturbação
PERT_VAL = 0.4  # magnitude da perturbação

def simular_com_perturbacao(func_ctrl, sp_seq, pert_k, pert_val, ku_change=None):
    """Simula com perturbação de carga e opcionalmente mudança de ganho."""
    global Ku
    Ku_original = Ku
    x = [0.0, 0.0]; y_hist = [0.0]; u_hist = []
    e_prev = 0.0; soma_e = 0.0

    for k in range(len(sp_seq)):
        # Mudança de parâmetro
        if ku_change and k == len(sp_seq)//2:
            Ku = Ku_original * ku_change

        sp = sp_seq[k]; e = sp - x[0]; de = e - e_prev; soma_e += e * dt
        u  = func_ctrl(e, de, soma_e, x[0], sp)

        # Perturbação de carga
        u_eff = u + (PERT_VAL if k == pert_k else 0.0)

        u_hist.append(u)
        x = plant_step(x, u_eff)
        y_hist.append(x[0])
        e_prev = e

    Ku = Ku_original   # restaurar
    return np.array(y_hist[1:]), np.array(u_hist)

pid_pert = PID(Kp_pid, Ki_pid, Kd_pid, dt)
pid_pert.reset()
def ctrl_pid_pert(e, de, soma_e, y, sp):
    return pid_pert.compute(sp, y)

y_pid_p,   u_pid_p   = simular_com_perturbacao(ctrl_pid_pert, sp_pert, PERT_K, PERT_VAL)
y_fuzzy_p, u_fuzzy_p = simular_com_perturbacao(ctrl_fuzzy,   sp_pert, PERT_K, PERT_VAL)
y_ann_p,   u_ann_p   = simular_com_perturbacao(ctrl_ann,     sp_pert, PERT_K, PERT_VAL)

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

ax = axes[0]
ax.plot(t_pert, sp_pert,    color='k',       lw=2, ls='--', label='Setpoint')
ax.plot(t_pert, y_pid_p,   color=COR_PID,   lw=2, label='PID')
ax.plot(t_pert, y_fuzzy_p, color=COR_FUZZY, lw=2, label='Fuzzy')
ax.plot(t_pert, y_ann_p,   color=COR_ANN,   lw=2, label='ANN')
ax.axvline(PERT_K*dt, color=COR_PERT, lw=2, ls=':', label=f'Perturbação (t={PERT_K*dt}s)')
ax.annotate(f'Perturbação\n+{PERT_VAL} na entrada',
            xy=(PERT_K*dt, 1.0), xytext=(PERT_K*dt+10, 1.25),
            arrowprops=dict(arrowstyle='->', color=COR_PERT),
            color=COR_PERT, fontsize=10)
ax.fill_between(t_pert, 0.95, 1.05, alpha=0.1, color='green', label='±5%')
ax.set_ylabel('Saída y(t)'); ax.set_title('Teste de Perturbação de Carga')
ax.legend(fontsize=10); ax.set_xlim(0, t_pert[-1])

ax = axes[1]
ax.step(t_pert, u_pid_p,   color=COR_PID,   lw=1.8, label='u PID',   where='post')
ax.step(t_pert, u_fuzzy_p, color=COR_FUZZY, lw=1.8, label='u Fuzzy', where='post')
ax.step(t_pert, u_ann_p,   color=COR_ANN,   lw=1.8, label='u ANN',   where='post')
ax.axvline(PERT_K*dt, color=COR_PERT, lw=2, ls=':')
ax.set_ylabel('u(t)'); ax.set_xlabel('Tempo (s)')
ax.set_title('Ação de Controle após Perturbação')
ax.legend(fontsize=10); ax.set_xlim(0, t_pert[-1])

plt.suptitle('Robustez a Perturbações de Carga', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ── Teste 2: Variação do ganho da planta (Ku muda +30%) ──────────────────────
pid_ku = PID(Kp_pid, Ki_pid, Kd_pid, dt); pid_ku.reset()
def ctrl_pid_ku(e, de, soma_e, y, sp):
    return pid_ku.compute(sp, y)

y_pid_ku,   _ = simular_com_perturbacao(ctrl_pid_ku, sp_pert, -1, 0, ku_change=1.3)
y_fuzzy_ku, _ = simular_com_perturbacao(ctrl_fuzzy,  sp_pert, -1, 0, ku_change=1.3)
y_ann_ku,   _ = simular_com_perturbacao(ctrl_ann,    sp_pert, -1, 0, ku_change=1.3)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(t_pert, sp_pert,    color='k',       lw=2, ls='--', label='Setpoint')
ax.plot(t_pert, y_pid_ku,   color=COR_PID,   lw=2, label='PID')
ax.plot(t_pert, y_fuzzy_ku, color=COR_FUZZY, lw=2, label='Fuzzy')
ax.plot(t_pert, y_ann_ku,   color=COR_ANN,   lw=2, label='ANN')
ax.axvline(N_PERT//2 * dt, color=COR_PERT, lw=2, ls=':',
           label=f'Ku muda +30% (t={N_PERT//2*dt}s)')
ax.annotate('Ku: 1.5 → 1.95\n(planta muda!)',
            xy=(N_PERT//2*dt, 1.0), xytext=(N_PERT//2*dt + 15, 1.25),
            arrowprops=dict(arrowstyle='->', color=COR_PERT),
            color=COR_PERT, fontsize=10)
ax.fill_between(t_pert, 0.95, 1.05, alpha=0.08, color='green')
ax.set_xlabel('Tempo (s)'); ax.set_ylabel('Saída y(t)')
ax.set_title('Robustez à Variação de Parâmetro da Planta (Ku +30%)', fontsize=13)
ax.legend(fontsize=10); ax.set_xlim(0, t_pert[-1])
plt.tight_layout(); plt.show()

print('Observação: a ANN foi treinada com Ku=1.5.')
print('Quando Ku muda, ela não foi re-treinada — isso testa robustez à incerteza.')

---
# 8. Métricas de Desempenho

## 8.1 Métricas de Controle

| Métrica | Fórmula | Interpretação |
|---|---|---|
| **ISE** | $\int e^2\,dt$ | Penaliza erros grandes — bom para controle rígido |
| **IAE** | $\int|e|\,dt$ | Erro absoluto médio — mais intuitivo |
| **ITAE** | $\int t\,|e|\,dt$ | Penaliza erros que persistem por muito tempo |
| **Overshoot** | $\frac{y_{max}-r}{r} \times 100\%$ | Ultrapassagem do setpoint |
| **TV** | $\sum|\Delta u|$ | Total Variation — suavidade da ação de controle |

In [ ]:
def calcular_metricas(y, u, sp, dt, nome):
    e   = sp - y
    t   = np.arange(len(e)) * dt
    ise  = np.sum(e**2) * dt
    iae  = np.sum(np.abs(e)) * dt
    itae = np.sum(t * np.abs(e)) * dt
    # Overshoot por degrau
    sp_changes = np.where(np.diff(sp) != 0)[0]
    overshoots = []
    for k_d in sp_changes:
        trecho = y[k_d:k_d+80]
        sp_d   = sp[k_d+1]
        if sp_d > sp[k_d]:  # degrau positivo
            os = (trecho.max() - sp_d) / abs(sp_d) * 100
        else:
            os = (sp_d - trecho.min()) / abs(sp_d) * 100
        overshoots.append(max(os, 0))
    tv = np.sum(np.abs(np.diff(u)))
    return {
        'Controlador': nome,
        'ISE':  round(ise, 4),
        'IAE':  round(iae, 4),
        'ITAE': round(itae, 4),
        'Overshoot médio (%)': round(np.mean(overshoots), 2),
        'TV (variação controle)': round(tv, 3),
    }

# Calcular métricas para os três controladores
met_pid   = calcular_metricas(y_pid,   u_pid,   sp_seq, dt, 'PID')
met_fuzzy = calcular_metricas(y_fuzzy, u_fuzzy, sp_seq, dt, 'Fuzzy')
met_ann   = calcular_metricas(y_ann,   u_ann,   sp_seq, dt, 'ANN')

df_met = pd.DataFrame([met_pid, met_fuzzy, met_ann]).set_index('Controlador')
print('=' * 65)
print('           TABELA DE DESEMPENHO — CENÁRIO NOMINAL')
print('=' * 65)
print(df_met.to_string())
print('=' * 65)

# Identificar melhor em cada métrica
print('\nMelhor desempenho por métrica:')
for col in df_met.columns:
    melhor = df_met[col].idxmin()
    print(f'  {col:<30}: {melhor} ({df_met.loc[melhor, col]:.4f})')

In [ ]:
# ── Dashboard visual de métricas ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
controladores = ['PID', 'Fuzzy', 'ANN']
cores_bar     = [COR_PID, COR_FUZZY, COR_ANN]
metricas_plot = ['ISE', 'IAE', 'ITAE', 'Overshoot médio (%)', 'TV (variação controle)']

for ax, metrica in zip(axes.flatten(), metricas_plot):
    vals = [df_met.loc[c, metrica] for c in controladores]
    bars = ax.bar(controladores, vals, color=cores_bar, edgecolor='white',
                  alpha=0.85, width=0.5)
    # Destaque do melhor
    idx_min = np.argmin(vals)
    bars[idx_min].set_edgecolor('gold'); bars[idx_min].set_linewidth(3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(metrica, fontsize=12)
    ax.set_ylim(0, max(vals) * 1.25)
    ax.tick_params(axis='x', labelsize=11)

# Radar chart (último painel)
ax_radar = axes[1, 2]
metricas_norm = df_met.copy()
for col in metricas_norm.columns:
    metricas_norm[col] = metricas_norm[col] / metricas_norm[col].max()

categorias = list(df_met.columns)
N_cat = len(categorias)
angles = [n / float(N_cat) * 2 * np.pi for n in range(N_cat)]
angles += angles[:1]

ax_radar.remove()
ax_radar = fig.add_subplot(2, 3, 6, polar=True)

for ctrl_nome, cor in zip(controladores, cores_bar):
    vals_r = metricas_norm.loc[ctrl_nome].tolist()
    vals_r += vals_r[:1]
    ax_radar.plot(angles, vals_r, color=cor, lw=2.5, label=ctrl_nome)
    ax_radar.fill(angles, vals_r, color=cor, alpha=0.1)

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(['ISE', 'IAE', 'ITAE', 'OS%', 'TV'], fontsize=10)
ax_radar.set_title('Radar — Normalizado\n(menor = melhor)', fontsize=11, pad=15)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)

# Legenda de ouro
patch_ouro = mpatches.Patch(edgecolor='gold', facecolor='none', lw=3, label='Melhor em cada métrica')
fig.legend(handles=[patch_ouro], loc='lower center', fontsize=10, ncol=1)

plt.suptitle('Dashboard de Desempenho — PID × Fuzzy × ANN', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
# 9. Resumo Final e Discussão

## 9.1 Tabela Comparativa Completa

| Aspecto | PID | Lógica Fuzzy | Rede Neural |
|---|---|---|---|
| **Base de projeto** | Modelo ou experimento | Regras do especialista | Dados históricos |
| **Conhecimento exigido** | Sintonia (Kp, Ki, Kd) | Regras SE-ENTÃO | Dataset de operação |
| **Tempo de projeto** | Rápido | Médio | Médio-longo (treino) |
| **Não-linearidades** | Limitado | Bom (regras adaptadas) | Muito bom (aprende) |
| **Interpretabilidade** | Alta | Alta | **Baixa (caixa preta)** |
| **Robustez** | Boa (quando bem sintonizado) | Boa | Depende do treino |
| **Adaptabilidade** | Manual (resintonizar) | Manual (alterar regras) | Re-treinar com novos dados |
| **Custo computacional** | Muito baixo | Baixo | Baixo (inferência) |

## 9.2 Quando usar cada um?

### Use PID quando:
- O processo é relativamente **linear** e bem comportado
- Há pouco tempo/recurso para projeto
- **Interpretabilidade** e facilidade de ajuste são importantes
- Regulatório e certificação exigem controlador auditável

### Use Lógica Fuzzy quando:
- Há **especialistas** que conhecem bem o processo mas não têm modelo matemático
- O processo é **não-linear** com comportamentos qualitativos claros
- Precisa de um controlador que possa ser **explicado em linguagem natural**
- Dados históricos são escassos

### Use Redes Neurais quando:
- Existem **muitos dados** de operação disponíveis
- O processo é complexo demais para derivar regras manualmente
- A performance é mais importante que a interpretabilidade
- O sistema muda e pode ser **re-treinado** periodicamente

## 9.3 Tendências Atuais

Na prática industrial moderna, os três são frequentemente **combinados**:
- **Fuzzy + PID:** ganhos do PID ajustados em tempo real por lógica fuzzy (Fuzzy-PID)
- **ANN + MPC:** rede neural como modelo interno do controlador preditivo (Neural MPC)
- **Reinforcement Learning:** agente aprende a lei de controle ótima por tentativa e erro

---

## 🧪 Exercícios Propostos

**Nível 1 — Exploração:**
1. Altere os ganhos do PID e observe o efeito no ISE e no overshoot
2. Adicione ou remova regras da base fuzzy — como isso afeta a superfície de controle?
3. Treine a ANN com apenas 5 cenários de setpoint. O desempenho piora?

**Nível 2 — Análise:**
4. Teste com perturbação de magnitude 1.0 (80% maior). Qual controlador é mais robusto?
5. Altere $\zeta = 0.1$ (sistema muito oscilatório). Os controladores precisam ser reprojetados?
6. Implemente um **Fuzzy-PID**: use a saída fuzzy para ajustar $K_p$ dinamicamente

**Nível 3 — Desafio:**
7. Treine a ANN com dados do **controlador fuzzy** (em vez do PID). O que muda?
8. Implemente uma rede neural que ajuste os ganhos do PID em tempo real (Neural Auto-Tuner)
9. Adicione **ruído de medição** ($y_{medido} = y + \mathcal{N}(0, 0.01)$) e compare a sensibilidade dos três controladores